In [ ]:
import os
import sys
sys.path.append("../")
sys.path.append("../..")
sys.path.append("../../src")
import warnings
warnings.filterwarnings("ignore")
import pickle
import avici
import numpy as np
from src.tools.metric import get_compared_components, get_skeleton, metric_skeleton_level, metric_cpdag_level

In [ ]:
model = avici.load_pretrained(download="scm-v0")
# model = avici.load_pretrained(download="neurips-linear")
# model = avici.load_pretrained(download="neurips-rff")
# model = avici.load_pretrained(download="neurips-grn")

In [ ]:
exp_name = 'exp_200'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../exps_of_result/avici/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx!=4:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../baselines/exps_of_result/avici/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [[all_nodes.index(item) for item in sublist] for sublist in intervention_targets]
    
    fusion_data = data_obs
    for i in range(data_int.shape[0]):
        fusion_data = np.vstack((fusion_data, data_int[i]))
    
    fusion_interv_index = np.zeros((len(data_obs), len(all_nodes)))
    for i in range(data_int.shape[0]):
        index_matrix = np.zeros((data_int.shape[1], len(all_nodes)))
        for j in intervention_targets_index[i]:
            index_matrix[:, j] = 1
        fusion_interv_index = np.vstack((fusion_interv_index, index_matrix))
    
    try:
        # Run AVICI
        pred_I_CPDAG_know = model(fusion_data, fusion_interv_index, return_probs=False)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_1'
intervention_size_list = [1, 1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/avici/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/avici/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [[all_nodes.index(item) for item in sublist] for sublist in intervention_targets]
    
    fusion_data = data_obs
    for i in range(data_int.shape[0]):
        fusion_data = np.vstack((fusion_data, data_int[i]))
    
    fusion_interv_index = np.zeros((len(data_obs), len(all_nodes)))
    for i in range(data_int.shape[0]):
        index_matrix = np.zeros((data_int.shape[1], len(all_nodes)))
        for j in intervention_targets_index[i]:
            index_matrix[:, j] = 1
        fusion_interv_index = np.vstack((fusion_interv_index, index_matrix))
    
    try:
        # Run AVICI
        pred_I_CPDAG_know = model(fusion_data, fusion_interv_index, return_probs=False)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_2'
intervention_size_list = [1, 1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/avici/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/avici/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [[all_nodes.index(item) for item in sublist] for sublist in intervention_targets]
    
    fusion_data = data_obs
    for i in range(data_int.shape[0]):
        fusion_data = np.vstack((fusion_data, data_int[i]))
    
    fusion_interv_index = np.zeros((len(data_obs), len(all_nodes)))
    for i in range(data_int.shape[0]):
        index_matrix = np.zeros((data_int.shape[1], len(all_nodes)))
        for j in intervention_targets_index[i]:
            index_matrix[:, j] = 1
        fusion_interv_index = np.vstack((fusion_interv_index, index_matrix))
    
    try:
        # Run AVICI
        pred_I_CPDAG_know = model(fusion_data, fusion_interv_index, return_probs=False)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_5'
intervention_size_list = [1, 1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/avici/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/avici/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [[all_nodes.index(item) for item in sublist] for sublist in intervention_targets]
    
    fusion_data = data_obs
    for i in range(data_int.shape[0]):
        fusion_data = np.vstack((fusion_data, data_int[i]))
    
    fusion_interv_index = np.zeros((len(data_obs), len(all_nodes)))
    for i in range(data_int.shape[0]):
        index_matrix = np.zeros((data_int.shape[1], len(all_nodes)))
        for j in intervention_targets_index[i]:
            index_matrix[:, j] = 1
        fusion_interv_index = np.vstack((fusion_interv_index, index_matrix))
    
    try:
        # Run AVICI
        pred_I_CPDAG_know = model(fusion_data, fusion_interv_index, return_probs=False)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_8'
intervention_size_list = [1, 1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/avici/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../../baselines/exps_of_result/avici/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [[all_nodes.index(item) for item in sublist] for sublist in intervention_targets]
    
    fusion_data = data_obs
    for i in range(data_int.shape[0]):
        fusion_data = np.vstack((fusion_data, data_int[i]))
    
    fusion_interv_index = np.zeros((len(data_obs), len(all_nodes)))
    for i in range(data_int.shape[0]):
        index_matrix = np.zeros((data_int.shape[1], len(all_nodes)))
        for j in intervention_targets_index[i]:
            index_matrix[:, j] = 1
        fusion_interv_index = np.vstack((fusion_interv_index, index_matrix))
    
    try:
        # Run AVICI
        pred_I_CPDAG_know = model(fusion_data, fusion_interv_index, return_probs=False)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')